# Wild-type Keio single-cell PHATE

Cells were selected from the corrected stationary-phase single-cell file, filtered to C2_NC_ratio in [0.8,1.3], and balanced to the smallest selected well (NumPy generator seed0). Features with missing fraction <0.5 were median-imputed and standardized. PHATE used 20 neighbors, decay50, automatic diffusion time and seed0. Coordinates are supplied so the figures rerun without the large raw file.

In [ ]:
from pathlib import Path
import numpy as np
import numpy.typing
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

PACKAGE = Path.cwd()
if not (PACKAGE / "Datasets").is_dir():
    PACKAGE = PACKAGE.parent
DATASETS = PACKAGE / "Datasets"
if not DATASETS.is_dir():
    raise FileNotFoundError("Launch this notebook from the package root or Github folder.")
plt.rcParams.update({"font.family": "Arial", "text.usetex": False})

import seaborn as sns
sns.set_theme(style="white")
from matplotlib.gridspec import GridSpec
book = DATASETS / "Dataset S2.xlsx"
df_phate = pd.read_excel(book, sheet_name="PHATE_plotted")
df_phate["Plate_Well"] = df_phate.Metadata_plate + "_" + df_phate.Metadata_well
df_ds = df_phate.copy()

## Optional embedding reconstruction
Uses only the sampled feature matrix supplied in S2. The full upstream file is `260928_single-cell_profiles_keio_stat.parquet`, available on request; it is not required here.

In [ ]:
RECOMPUTE_PHATE = False
if RECOMPUTE_PHATE:
    import phate
    from sklearn.preprocessing import StandardScaler
    inputs = pd.read_excel(book, sheet_name="PHATE_input_cells")
    assert inputs.observation_id.equals(df_phate.observation_id)
    X = inputs.drop(columns="observation_id")
    X = X.fillna(X.median())
    coordinates = phate.PHATE(n_components=2, knn=20, decay=50, t="auto", random_state=0, verbose=0).fit_transform(StandardScaler().fit_transform(X))
    df_phate[["PHATE_1", "PHATE_2"]] = coordinates

In [ ]:
(fig, ax) = plt.subplots(figsize=(10, 8))
sns.scatterplot(data=df_phate, x='PHATE_1', y='PHATE_2', hue='Plate_Well', s=8, linewidth=0, alpha=0.7, palette='tab20', ax=ax)
ax.legend(title='Plate_Well', markerscale=3)
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
plate_wells = df_phate['Plate_Well'].unique()
(n_rows, n_cols) = (3, 5)
(fig, axes) = plt.subplots(n_rows, n_cols, figsize=(15, 9), sharex=True, sharey=True)
axes = axes.flatten()
for (ax, pw) in zip(axes, plate_wells):
    ax.scatter(df_phate['PHATE_1'], df_phate['PHATE_2'], c='#C0C0C0', s=4, alpha=0.3, linewidth=0, rasterized=True)
    mask = df_phate['Plate_Well'] == pw
    ax.scatter(df_phate.loc[mask, 'PHATE_1'], df_phate.loc[mask, 'PHATE_2'], c='#d62728', s=4, alpha=0.9, linewidth=0, rasterized=True)
    ax.set_title(pw, fontsize=9)
    ax.set_xticks([])
    ax.set_yticks([])
for ax in axes[len(plate_wells):]:
    ax.axis('off')
fig.suptitle('PHATE – each Plate_Well highlighted', fontsize=14)
fig.tight_layout(rect=[0, 0, 1, 0.95])
plt.show()

In [ ]:
from matplotlib.gridspec import GridSpec
panel_mm = 50
figsize_in = panel_mm / 25.4
fig = plt.figure(figsize=(figsize_in, figsize_in), dpi=300)
ax = fig.add_subplot(111)
sns.kdeplot(data=df_phate, x='PHATE_1', y='PHATE_2', levels=15, color='black', linewidths=0.5, ax=ax)
ax.scatter(df_phate['PHATE_1'], df_phate['PHATE_2'], s=3, c='steelblue', alpha=0.8, linewidth=0, rasterized=True)
ax.set_axis_off()
ax.set_aspect('equal', adjustable='box')
ax.margins(0)
plt.tight_layout(pad=0)
plt.show()

In [ ]:
import os
import re
import numpy as np
import matplotlib.pyplot as plt
panel_mm = 50
figsize_in = panel_mm / 25.4
x = df_phate['PHATE_1'].to_numpy()
y = df_phate['PHATE_2'].to_numpy()
(x_min, x_max) = (np.nanmin(x), np.nanmax(x))
(y_min, y_max) = (np.nanmin(y), np.nanmax(y))
x_pad = 0.02 * (x_max - x_min)
y_pad = 0.02 * (y_max - y_min)
xlim = (x_min - x_pad, x_max + x_pad)
ylim = (y_min - y_pad, y_max + y_pad)
plate_wells = df_phate['Plate_Well'].unique()
for pw in plate_wells:
    (fig, ax) = plt.subplots(figsize=(figsize_in, figsize_in), dpi=500)
    ax.scatter(df_phate['PHATE_1'], df_phate['PHATE_2'], c='#989898', s=3, alpha=0.5, linewidth=0, rasterized=True)
    mask = df_phate['Plate_Well'] == pw
    ax.scatter(df_phate.loc[mask, 'PHATE_1'], df_phate.loc[mask, 'PHATE_2'], c='#D55E00', s=3, alpha=0.9, linewidth=0, rasterized=True)
    ax.set_xlim(xlim)
    ax.set_ylim(ylim)
    ax.set_aspect('equal', adjustable='box')
    ax.set_axis_off()
    fig.subplots_adjust(left=0, right=1, bottom=0, top=1)
    plt.show()

## Feature overlays
Constriction is multiplied by 2 and clipped to [0,1] only in a display copy, with a fixed 0–1 colour scale. Other features retain the source 1st–99th percentile scaling. Circularity remains in the PHATE input matrix but is not plotted. Raw observations and embedding coordinates are unchanged.

In [ ]:
def plot_phate_highlight(df_phate: pd.DataFrame, df_numeric: pd.DataFrame, feature: str | None=None, plate: str | None=None, well: str | None=None, cmap: str='cividis', s_background: float=2.5, s_highlight: float=1.5, alpha_background: float=0.25, alpha_highlight: float=0.8, vmin_pct: float=1, vmax_pct: float=99):
    panel_mm = 50
    cbar_mm = 6
    width_in = panel_mm / 25.4
    height_in = (panel_mm + cbar_mm) / 25.4 if feature else panel_mm / 25.4
    fig = plt.figure(figsize=(width_in, height_in), dpi=300)
    if feature:
        gs = GridSpec(nrows=2, ncols=1, height_ratios=[panel_mm, cbar_mm], hspace=0.15)
        ax = fig.add_subplot(gs[0, 0])
        cax = fig.add_subplot(gs[1, 0])
    else:
        ax = fig.add_subplot(111)
        cax = None
    if plate is not None and well is not None:
        mask = df_phate['Metadata_plate'].eq(plate) & df_phate['Metadata_well'].eq(well)
    else:
        mask = pd.Series(True, index=df_phate.index)
    ax.scatter(df_phate.loc[~mask, 'PHATE_1'], df_phate.loc[~mask, 'PHATE_2'], c='#c0c0c0', s=s_background, alpha=alpha_background, linewidth=0, rasterized=True)
    if feature is not None:
        values = df_numeric.loc[df_phate.index, feature]
        if feature == 'cell_constriction_degree_width_axial':
            vmin, vmax = 0.0, 1.0
        else:
            vmin = np.nanpercentile(values, vmin_pct)
            vmax = np.nanpercentile(values, vmax_pct)
        sc = ax.scatter(df_phate.loc[mask, 'PHATE_1'], df_phate.loc[mask, 'PHATE_2'], c=values.loc[mask], cmap=cmap, vmin=vmin, vmax=vmax, s=s_highlight, alpha=alpha_highlight, linewidth=0, rasterized=True)
        cb = fig.colorbar(sc, cax=cax, orientation='horizontal')
        label = 'Constriction degree' if feature == 'cell_constriction_degree_width_axial' else f'{feature} ({vmin_pct}-{vmax_pct} percentile)'
        cb.set_label(label, fontsize=7)
        cb.ax.tick_params(labelsize=6)
    else:
        ax.scatter(df_phate.loc[mask, 'PHATE_1'], df_phate.loc[mask, 'PHATE_2'], c='#d62728', s=s_highlight, alpha=alpha_highlight, linewidth=0, rasterized=True)
    ax.set_axis_off()
    ax.set_aspect('equal', adjustable='box')
    plt.show()
df_ds_for_features = df_ds.copy()
if 'cell_constriction_degree_width_axial' in df_ds_for_features.columns:
    df_ds_for_features['cell_constriction_degree_width_axial'] = (df_ds_for_features['cell_constriction_degree_width_axial'] * 2).clip(0, 1)
features_to_plot = ['cell_length', 'cell_width', 'cell_constriction_degree_width_axial', 'C5_object_number', 'C5_NC_ratio', 'C4_cell_mesh_intensity_mean', 'C3_cell_mesh_intensity_mean', 'C2_cell_glcm_energy']
for feature_name in features_to_plot:
    if feature_name in df_ds_for_features.columns:
        plot_phate_highlight(df_phate=df_phate, df_numeric=df_ds_for_features, feature=feature_name)